### Configuration

In [3]:
import sys, random, collections
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.catalog import load_catalog
from src.dataset_builder import ejemplos_2a, Q_TEST

df = load_catalog(ROOT / "data" / "example_data.csv")
TIENDA = "TecnoBolivia"   # Store Name
rng = random.Random(42)

### Generate Examples

In [5]:
ej = ejemplos_2a(df, TIENDA, rng, keep_original=True)
print(len(ej), "ejemplos")
print(dict(collections.Counter(e["meta"]["intent"] for e in ej)), "\n")
for e in ej[:8]:
    pregunta = e["messages"][1]["content"].split("PREGUNTA:")[1].strip()
    print("👤", pregunta)
    print("🤖", e["messages"][2]["content"], "\n")

132 ejemplos
{'marca': 7, 'cat_budget': 31, 'cat_attr': 14, 'cat_only': 20, 'ficha': 14, 'specs': 13, 'stock': 8, 'cat_cheapest': 16, 'precio': 9} 

👤 ¿Tienen algo de MSI?
🤖 Por ahora no tenemos productos MSI disponibles 😕. 

👤 Busco teléfonos que no pase de 1500
🤖 Esto es lo que tenemos disponible 😊
- Tecno Spark 20: 900 Bs (20 en stock)
- Infinix Hot 40 Pro: 1.100 Bs (9 en stock)
- Motorola Moto G54: 1.500 Bs (7 en stock) 

👤 ¿Tienen smartphone que tengan pantalla pOLED 144 Hz?
🤖 Esto es lo que tenemos disponible 😊
- Motorola Edge 40 Neo: 3.300 Bs (3 en stock) 

👤 Mándame opciones de compu
🤖 Esto es lo que tenemos disponible 😊
- HP 15-fd0: 3.900 Bs (5 en stock)
- Lenovo IdeaPad 3: 4.500 Bs (8 en stock)
- Acer Aspire 5: 4.800 Bs (6 en stock) 

👤 ¿Qué audífono venden?
🤖 Esto es lo que tenemos disponible 😊
- Xiaomi Redmi Buds 4 Lite: 200 Bs (25 en stock)
- JBL Tune 510BT: 450 Bs (18 en stock)
- Logitech H390: 550 Bs (9 en stock) 

👤 ¿Tienen audífono de menos de 1500?
🤖 Esto es lo que te

### Examples based on Budget

In [6]:
ej = ejemplos_2a(df, TIENDA, rng, keep_original=True)
print(len(ej), "ejemplos")
print(dict(collections.Counter(e["meta"]["intent"] for e in ej)), "\n")
for e in ej[:8]:
    pregunta = e["messages"][1]["content"].split("PREGUNTA:")[1].strip()
    print("👤", pregunta)
    print("🤖", e["messages"][2]["content"], "\n")

132 ejemplos
{'cat_attr': 21, 'marca': 13, 'cat_budget': 34, 'stock': 10, 'precio': 10, 'ficha': 20, 'cat_cheapest': 10, 'cat_only': 10, 'specs': 4} 

👤 ¿qué accesorios traen para laptop hasta 15.6"?
🤖 Por ahora no tenemos accesorios con Para laptop hasta 15.6" disponibles 😕. 

👤 ¿Qué hay de ZTE?
🤖 Esto es lo que tenemos disponible 😊
- ZTE Blade A54: 700 Bs (11 en stock) 

👤 ¿Qué pantallas tienen que cuesten menos de 2.150 Bs?
🤖 Esto es lo que tenemos disponible 😊
- AOC 22B1HS: 800 Bs (14 en stock)
- LG 24MP400: 1.100 Bs (8 en stock)
- Dell P2422H: 1.500 Bs (5 en stock) 

👤 computadora hasta 5500 Bs
🤖 Esto es lo que tenemos disponible 😊
- ASUS Vivobook Go 14: 3.600 Bs (12 en stock)
- HP 15-fd0: 3.900 Bs (5 en stock)
- Acer Aspire 5: 4.800 Bs (6 en stock) 

👤 ¿Qué hay de Acer?
🤖 Esto es lo que tenemos disponible 😊
- Acer Aspire 5: 4.800 Bs (6 en stock) 

👤 ¿Les queda hp deskjet 2775?
🤖 De HP DeskJet 2775 tenemos 6 unidades disponibles 😊 

👤 Tienen algo de Canon
🤖 Por ahora no tenemos pr

### Test Questions

In [7]:
for e in ejemplos_2a(df, TIENDA, random.Random(3), banco=Q_TEST)[:6]:
    print("👤", e["messages"][1]["content"].split("PREGUNTA:")[1].strip())

👤 Mouse Logitech M170 ¿lo tienen disponible?
👤 Epson L3250 ¿lo tienen disponible?
👤 Tienen unidades de Hub USB-C Ugreen 7 en 1?
👤 Mi plata llega a 3750 Bs, ¿qué monitor hay?
👤 celulares más económicos que haya
👤 Quisiera saber si hay Hub USB-C Ugreen 7 en 1


### Test Different catalog versions

In [8]:
import re

rng_val = random.Random(7)
todos = []
for v in range(15):
    todos += ejemplos_2a(df, TIENDA, rng_val, keep_original=(v == 0))

c = collections.Counter(e["meta"]["intent"] for e in todos)
n_desc = sum(n for k, n in c.items() if k.startswith("cat_") or k == "marca")
print(f"{len(todos)} ejemplos | descubrimiento: {n_desc/len(todos):.0%}")

def sin_respaldo(e):
    """Números de la respuesta que no aparecen en el contexto ni en la pregunta."""
    user = e["messages"][1]["content"].replace(".", "").replace(",", "")
    user += " " + str(e["meta"].get("presupuesto", ""))
    resp = e["messages"][2]["content"]
    return [n for n in re.findall(r"\d+(?:[.,]\d+)*", resp)
            if n.replace(".", "").replace(",", "") not in user]

malos = [e for e in todos if sin_respaldo(e)]
print("Ejemplos con números inventados:", len(malos))

preguntas = [e["messages"][1]["content"].split("PREGUNTA:")[1].strip() for e in todos]
print("Preguntas distintas:", len(set(preguntas)), "de", len(preguntas))

1980 ejemplos | descubrimiento: 67%
Ejemplos con números inventados: 0
Preguntas distintas: 1770 de 1980
